In [ ]:
import numpy as np
import torch
from monai.data import Dataset, DataLoader, decollate_batch
from monai.inferers import sliding_window_inference
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.transforms import AsDiscrete, Compose, EnsureTyped, RandFlipd
from monai.utils import set_determinism


In [ ]:

set_determinism(seed=0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIZE = 64

def make_sample(rng):
    """Random noisy volume with one bright sphere; label marks the sphere."""
    zz, yy, xx = np.mgrid[:SIZE, :SIZE, :SIZE]
    c = rng.integers(16, SIZE - 16, size=3)
    r = rng.integers(6, 14)
    mask = ((zz - c[0]) ** 2 + (yy - c[1]) ** 2 + (xx - c[2]) ** 2) < r**2
    img = rng.normal(0.0, 0.5, (SIZE,) * 3) + mask * 1.0
    return {
        "image": img[None].astype(np.float32),  # channel-first: (1, D, H, W)
        "label": mask[None].astype(np.float32),
    }


rng = np.random.default_rng(0)
train_data = [make_sample(rng) for _ in range(40)]
val_data = [make_sample(rng) for _ in range(8)]

train_tf = Compose([
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    EnsureTyped(keys=["image", "label"]),
])
val_tf = Compose([EnsureTyped(keys=["image", "label"])])

train_loader = DataLoader(Dataset(train_data, train_tf), batch_size=4, shuffle=True)
val_loader = DataLoader(Dataset(val_data, val_tf), batch_size=1)

model = UNet(
    spatial_dims=3, in_channels=1, out_channels=2,
    channels=(8, 16, 32, 64), strides=(2, 2, 2), num_res_units=1,
).to(device)

loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
dice_metric = DiceMetric(include_background=False, reduction="mean")
post_pred = AsDiscrete(argmax=True, to_onehot=2)
post_label = AsDiscrete(to_onehot=2)


In [ ]:
EPOCHS = 30
best = 0.0
for epoch in range(1, EPOCHS + 1):
    # ---- train ----
    model.train()
    total = 0.0
    for batch in train_loader:
        x, y = batch["image"].to(device), batch["label"].to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item()

    # ---- validate (sliding window over full volumes) ----
    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            for batch in val_loader:
                x, y = batch["image"].to(device), batch["label"].to(device)
                out = sliding_window_inference(x, roi_size=(32, 32, 32),
                                               sw_batch_size=4, predictor=model)
                preds = [post_pred(p) for p in decollate_batch(out)]
                labels = [post_label(l) for l in decollate_batch(y)]
                dice_metric(y_pred=preds, y=labels)
        dice = dice_metric.aggregate().item()
        dice_metric.reset()
        print(f"epoch {epoch:02d}  loss {total / len(train_loader):.4f}  val Dice {dice:.4f}")
        if dice > best:
            best = dice
            torch.save(model.state_dict(), "best_model.pth")

print(f"Done. Best val Dice: {best:.4f} (weights saved to best_model.pth)")